# 07 Final Epoch Calibration, Train+Val Refit, and Test

**Objective:** calibrate the stopping epoch for Notebook 06's locked
model/augmentation/resolution on disjoint train and validation data, restart
from pretrained weights, refit on train + validation for exactly that epoch
count, and evaluate the independent test set once.

**Success criteria:** validation alone selects the epoch, test never affects
that choice, the final artifact is refit `last.pt`, and all decision metadata
remains auditable.


## Setup and one-time test guard

`DRY_RUN=True` validates both training phases without training or evaluating
test. For an intentional final run, set `DRY_RUN=False` and
`ALLOW_TEST_EVALUATION=True`. `SOURCE_RESOLUTION_RUN` may pin a completed
Notebook 06 artifact. An experiment override must match the experiment already
calibrated in Notebook 06.


In [ ]:
# Purpose: initialize the final two-phase workflow from the complete upstream decision chain.
# Inputs: Notebook 06 metadata, its exact augmentation artifact, final runtime config, and explicit test authorization.
# Guard: a real run requires completed resolution metadata; test reporting is authorized before any final fitting begins.
# Output: one immutable final artifact with locked model, experiment, resolution, and provenance.
from pathlib import Path
import sys

BOOTSTRAP_ROOT = Path.cwd().resolve()
if BOOTSTRAP_ROOT.name == "notebooks":
    BOOTSTRAP_ROOT = BOOTSTRAP_ROOT.parent
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from src.artifacts import (
    create_run,
    find_latest_run,
    open_run,
    read_run_metadata,
    runtime_metadata,
    update_run_metadata,
    write_run_metadata,
)
from src.config import (
    find_project_root,
    get_experiment,
    load_pipeline_config,
    online_augmentation_args,
    resolve_final_experiment,
)
from src.dataset.manifests import (
    build_refit_training_manifest,
    copy_manifest,
    experiment_manifest,
    manifest_pair_content_sha256,
    manifest_sha256,
    read_manifest,
    write_ultralytics_dataset_yaml,
)
from src.training.evaluate_yolo import (
    evaluate_model,
    extract_best_epoch,
    save_confusion_matrix,
    save_prediction_grid,
)
from src.training.export_model import export_final_model
from src.training.train_yolo import (
    build_train_args,
    clean_training_intermediates,
    format_progress_event,
    retain_training_checkpoint,
    run_training_specs,
)

V2_ROOT = find_project_root(BOOTSTRAP_ROOT)
CONFIG = load_pipeline_config(V2_ROOT)
TRAINING_CONFIG = CONFIG["training"]
SEED = int(TRAINING_CONFIG["seed"])

SOURCE_RESOLUTION_RUN = None
SOURCE_AUGMENTATION_RUN = None
# Permitted only when Notebook 06 calibrated the same experiment.
FINAL_EXPERIMENT_OVERRIDE = None
DRY_RUN = True
ALLOW_TEST_EVALUATION = False
if not DRY_RUN and not ALLOW_TEST_EVALUATION:
    raise RuntimeError(
        "Final calibration/refit/test requires ALLOW_TEST_EVALUATION=True"
    )

RESOLUTION_RUN = (
    open_run(SOURCE_RESOLUTION_RUN)
    if SOURCE_RESOLUTION_RUN
    else find_latest_run(
        V2_ROOT,
        "resolution-calibration",
        "csv/resolution_calibration_results.csv",
    )
)
resolution_metadata = read_run_metadata(RESOLUTION_RUN)
if not DRY_RUN and resolution_metadata.get("status") != "completed":
    raise RuntimeError(f"Resolution run is not completed: {RESOLUTION_RUN.run_id}")

selected_model = resolution_metadata.get("selected_model")
resolution_selected_experiment = resolution_metadata.get("selected_experiment")
selected_resolution_profile = resolution_metadata.get("selected_resolution_profile")
selected_imgsz = resolution_metadata.get("selected_imgsz")
selected_batch = resolution_metadata.get("selected_batch")
if DRY_RUN:
    # Fallbacks validate wiring only and cannot authorize a real final run.
    selected_model = selected_model or TRAINING_CONFIG["candidate_models"][0]
    resolution_selected_experiment = (
        resolution_selected_experiment or TRAINING_CONFIG["candidate_experiment"]
    )
    selected_resolution_profile = selected_resolution_profile or "dry-run-baseline"
    selected_imgsz = selected_imgsz or TRAINING_CONFIG["imgsz"]
    selected_batch = selected_batch or TRAINING_CONFIG["batch"]
if not all(
    (
        selected_model,
        resolution_selected_experiment,
        selected_resolution_profile,
        selected_imgsz,
        selected_batch,
    )
):
    raise RuntimeError(
        "Completed Notebook 06 metadata must lock model, experiment, imgsz, and batch"
    )
selected_imgsz = int(selected_imgsz)
selected_batch = int(selected_batch)

selected_experiment, experiment_selection_source = resolve_final_experiment(
    CONFIG,
    str(resolution_selected_experiment),
    str(FINAL_EXPERIMENT_OVERRIDE) if FINAL_EXPERIMENT_OVERRIDE else None,
)
if selected_experiment != str(resolution_selected_experiment):
    raise RuntimeError(
        "Final experiment differs from Notebook 06 calibration; rerun Notebook 06 "
        "with the intended RESOLUTION_EXPERIMENT_OVERRIDE before test access"
    )
experiment_selection_source = (
    "manual_override_matching_resolution"
    if FINAL_EXPERIMENT_OVERRIDE
    else "resolution_calibration"
)

upstream_augmentation_id = resolution_metadata.get("source_augmentation_run")
if SOURCE_AUGMENTATION_RUN:
    AUGMENTATION_RUN = open_run(SOURCE_AUGMENTATION_RUN)
elif upstream_augmentation_id:
    AUGMENTATION_RUN = open_run(V2_ROOT / "artifacts" / str(upstream_augmentation_id))
else:
    raise RuntimeError("Resolution metadata does not identify its augmentation run")
augmentation_metadata = read_run_metadata(AUGMENTATION_RUN)
expected_test_content_sha256 = augmentation_metadata.get("test_pair_content_sha256")
if not expected_test_content_sha256:
    raise RuntimeError(
        "Upstream augmentation metadata lacks the frozen test content fingerprint"
    )

RUN = create_run(
    V2_ROOT,
    "final",
    (
        f"{Path(str(selected_model)).stem}-{selected_experiment}-"
        f"{selected_resolution_profile}-calibrated-refit"
    ),
    SEED,
)
for manifest_name in (
    "train_original.txt",
    "train_original_plus_offline.txt",
    "val.txt",
    "test.txt",
):
    copy_manifest(
        AUGMENTATION_RUN.manifests / manifest_name,
        RUN.manifests / manifest_name,
    )
write_run_metadata(
    RUN,
    {
        "stage": "final",
        "status": "started",
        "dry_run": DRY_RUN,
        "test_evaluation_authorized": ALLOW_TEST_EVALUATION,
        "seed": SEED,
        "selected_model": str(selected_model),
        "selected_experiment": selected_experiment,
        "selected_resolution_profile": str(selected_resolution_profile),
        "selected_imgsz": selected_imgsz,
        "selected_batch": selected_batch,
        "epoch_policy": "calibrate_on_train_val_then_refit_train_plus_val",
        "final_experiment_override": FINAL_EXPERIMENT_OVERRIDE,
        "experiment_selection_source": experiment_selection_source,
        "source_resolution_run": RESOLUTION_RUN.run_id,
        "source_ablation_run": resolution_metadata.get("source_ablation_run"),
        "source_candidate_run": resolution_metadata.get("source_candidate_run"),
        "source_augmentation_run": AUGMENTATION_RUN.run_id,
        "runtime": runtime_metadata(V2_ROOT),
    },
)


## Materialize calibration, refit, and test membership

Calibration trains on the selected A-D train manifest and validates on the
clean frozen validation set. Refit adds those validation images to the
selected training membership, preserving offline samples when experiment C or
D was selected. Both training YAMLs omit test; the evaluation YAML is created
only after refit.


In [ ]:
# Purpose: create disjoint calibration membership, train-plus-val refit membership, and verify test integrity.
# Inputs: selected experiment, copied original/combined manifests, frozen validation/test, and Notebook 02 fingerprint.
# Guard: preserve selected offline data, reject train/val overlap, omit test from both training YAMLs, and abort on test-byte drift.
# Output: two training YAMLs plus compact membership and locked-setting summaries.
experiment = get_experiment(CONFIG, selected_experiment)
calibration_train_manifest = experiment_manifest(
    experiment,
    RUN.manifests / "train_original.txt",
    RUN.manifests / "train_original_plus_offline.txt",
)
calibration_val_manifest = RUN.manifests / "val.txt"
test_manifest = RUN.manifests / "test.txt"

# Verify both test images and labels without computing any test metric.
test_content_sha256 = manifest_pair_content_sha256(test_manifest)
if test_content_sha256 != expected_test_content_sha256:
    raise RuntimeError("Untouched test image/label content changed after split freezing")

refit_training_manifest = build_refit_training_manifest(
    calibration_train_manifest,
    calibration_val_manifest,
    RUN.manifests / "refit_train_plus_val.txt",
)
calibration_dataset_yaml = write_ultralytics_dataset_yaml(
    RUN.root / "calibration_dataset.yaml",
    calibration_train_manifest,
    calibration_val_manifest,
    CONFIG["classes"],
)
# Ultralytics requires a val key even when refit validation is disabled.
refit_dataset_yaml = write_ultralytics_dataset_yaml(
    RUN.root / "refit_dataset.yaml",
    refit_training_manifest,
    refit_training_manifest,
    CONFIG["classes"],
    allow_train_val_overlap=True,
)

membership_df = pd.DataFrame(
    [
        {
            "phase": "calibration train",
            "manifest": calibration_train_manifest.name,
            "images": len(read_manifest(calibration_train_manifest)),
        },
        {
            "phase": "calibration val",
            "manifest": calibration_val_manifest.name,
            "images": len(read_manifest(calibration_val_manifest)),
        },
        {
            "phase": "locked-epoch refit",
            "manifest": refit_training_manifest.name,
            "images": len(read_manifest(refit_training_manifest)),
        },
        {
            "phase": "untouched test",
            "manifest": test_manifest.name,
            "images": len(read_manifest(test_manifest)),
        },
    ]
)
locked_settings_df = pd.DataFrame(
    [
        {
            "model": str(selected_model),
            "experiment": selected_experiment,
            "resolution_profile": str(selected_resolution_profile),
            "imgsz": selected_imgsz,
            "batch": selected_batch,
            "online_augmentation": bool(experiment["online_augmentation"]),
            "calibration_max_epochs": int(TRAINING_CONFIG["final_epochs"]),
            "calibration_patience": int(TRAINING_CONFIG["patience"]),
            "dry_run": DRY_RUN,
        }
    ]
)
display(locked_settings_df)
display(membership_df)


#### Observation

TODO after execution:

- Confirm model, experiment, resolution, and batch exactly match Notebook 06.
- Confirm refit membership equals selected train plus clean validation.
- Confirm both training YAMLs omit test and the test fingerprint is unchanged.


## Phase 1 — calibrate the stopping epoch on validation

Train the selected model from its pretrained checkpoint on calibration train.
Validation runs every epoch with configured patience. Lock the first epoch
with maximum validation mAP50-95, matching Ultralytics detection fitness. Test
metrics remain unavailable.


In [ ]:
# Purpose: estimate the stopping epoch for the exact locked model, data policy, augmentation, and resolution.
# Inputs: disjoint calibration train/val YAML and Notebook 06 imgsz/batch with final epoch budget.
# Guard: train one model only, permit early stopping, and keep test absent from the dataset YAML.
# Output: one live validation panel and a compact best-epoch selection record.
calibration_train_args = build_train_args(TRAINING_CONFIG, "final_epochs")
calibration_train_args.update(
    {"imgsz": selected_imgsz, "batch": selected_batch}
)
calibration_spec = {
    "run_name": (
        f"final-calibration-{Path(str(selected_model)).stem}-"
        f"{selected_experiment}-{selected_resolution_profile}"
    ),
    "model_name": str(selected_model),
    "experiment": selected_experiment,
    "data_yaml": calibration_dataset_yaml,
    "online_augmentation": bool(experiment["online_augmentation"]),
    "augmentation_args": online_augmentation_args(
        CONFIG["augmentation"],
        enabled=bool(experiment["online_augmentation"]),
    ),
}
calibration_progress = display(
    Markdown("Final epoch calibration is waiting to start."),
    display_id=True,
)


def update_calibration_progress(event: dict) -> None:
    """Render one compact calibration event in the existing panel."""
    calibration_progress.update(Markdown(format_progress_event(event)))


calibration_summary_df = run_training_specs(
    [calibration_spec],
    RUN.models,
    calibration_train_args,
    dry_run=DRY_RUN,
    continue_on_error=False,
    progress_callback=update_calibration_progress,
)
if DRY_RUN:
    epoch_selection = {
        "best_epoch": int(TRAINING_CONFIG["final_epochs"]),
        "selection_metric": "dry_run_fallback",
        "precision": None,
        "recall": None,
        "map50": None,
        "map50_95": None,
    }
    epoch_selection_source = "dry_run_fallback"
else:
    calibration_run_dir = Path(str(calibration_summary_df.iloc[0]["run_dir"]))
    epoch_selection = extract_best_epoch(calibration_run_dir)
    epoch_selection_source = "calibration_validation_map50_95"

locked_refit_epochs = int(epoch_selection["best_epoch"])
for key, value in epoch_selection.items():
    calibration_summary_df.loc[0, f"selected_{key}"] = value
calibration_summary_path = RUN.csv / "final_calibration_epoch_selection.csv"
calibration_summary_df.to_csv(calibration_summary_path, index=False)
display(
    calibration_summary_df[
        [
            "model_name",
            "status",
            "selected_best_epoch",
            "selected_recall",
            "selected_map50",
            "selected_map50_95",
            "training_time_seconds",
        ]
    ]
)


#### Observation

TODO after execution:

- Record the validation-selected best epoch and mAP50-95.
- Note whether early stopping shortened the maximum epoch budget.
- Do not alter this epoch after seeing any test result.


## Phase 2 — refit train + val for the locked epoch count

Restart the same architecture from its original pretrained checkpoint; do not
continue from calibration `best.pt`. Train all selected development examples
for exactly `locked_refit_epochs`. Validation and early stopping are disabled,
so the final retained checkpoint is refit `last.pt`.


In [ ]:
# Purpose: refit from pretrained initialization on selected train plus validation for the locked epoch count.
# Inputs: refit manifest, validation-selected epoch, Notebook 06 resolution, and identical augmentation policy.
# Guard: restart from selected_model, disable validation/early stopping, and never change epochs from test evidence.
# Output: one loss-only progress panel and a compact fixed-epoch refit summary.
refit_train_args = build_train_args(TRAINING_CONFIG, "final_epochs")
refit_train_args.update(
    {
        "epochs": locked_refit_epochs,
        "imgsz": selected_imgsz,
        "batch": selected_batch,
        "val": False,
        "patience": 0,
    }
)
refit_spec = {
    "run_name": (
        f"final-refit-{Path(str(selected_model)).stem}-"
        f"{selected_experiment}-{selected_resolution_profile}-e{locked_refit_epochs}"
    ),
    "model_name": str(selected_model),
    "experiment": selected_experiment,
    "data_yaml": refit_dataset_yaml,
    "online_augmentation": bool(experiment["online_augmentation"]),
    "augmentation_args": online_augmentation_args(
        CONFIG["augmentation"],
        enabled=bool(experiment["online_augmentation"]),
    ),
}
refit_progress = display(
    Markdown("Locked-epoch final refit is waiting to start."),
    display_id=True,
)


def update_refit_progress(event: dict) -> None:
    """Render one compact loss-only refit event in the existing panel."""
    refit_progress.update(Markdown(format_progress_event(event)))


refit_summary_df = run_training_specs(
    [refit_spec],
    RUN.models,
    refit_train_args,
    dry_run=DRY_RUN,
    continue_on_error=False,
    progress_callback=update_refit_progress,
)
refit_summary_df.insert(2, "locked_refit_epochs", locked_refit_epochs)
refit_summary_path = RUN.csv / "final_locked_epoch_refit_summary.csv"
refit_summary_df.to_csv(refit_summary_path, index=False)
display(
    refit_summary_df[
        [
            "model_name",
            "status",
            "locked_refit_epochs",
            "training_time_seconds",
            "model_size_mb",
            "error_message",
        ]
    ]
)


#### Observation

TODO after execution:

- Confirm refit completed exactly the validation-locked epoch count.
- Treat any metric over refit membership as diagnostic only; it did not select the checkpoint.


## Retain refit `last.pt` and evaluate test once

Create the test-enabled evaluation YAML only after refit, copy refit `last.pt`
to a descriptive stable model file, and evaluate `split=test`. Test metrics are
reporting-only and cannot change model, data policy, resolution, or epoch count.


In [ ]:
# Purpose: retain the fixed-epoch refit model and perform the authorized untouched-test evaluation.
# Inputs: successful refit last.pt, test manifest, selected resolution/device, and reporting confidence.
# Guard: dry-run skips test evaluation; only this post-refit cell creates a YAML containing the test key.
# Output: retained final model, compact test CSVs, confusion matrix, and six-image prediction grid.
final_checkpoint = None
evaluation_dataset_yaml = None
final_overall_df = pd.DataFrame()
final_class_metrics_df = pd.DataFrame()
confusion_figure_path = None
prediction_figure_path = None
if DRY_RUN:
    print("Dry run: checkpoint retention and test evaluation skipped.")
else:
    final_checkpoint = retain_training_checkpoint(
        refit_summary_df,
        RUN.models
        / (
            f"{Path(str(selected_model)).stem}_{selected_experiment}_"
            f"{selected_resolution_profile}_e{locked_refit_epochs}_train-plus-val.pt"
        ),
        checkpoint_kind="last",
    )
    evaluation_dataset_yaml = write_ultralytics_dataset_yaml(
        RUN.root / "evaluation_dataset.yaml",
        calibration_train_manifest,
        calibration_val_manifest,
        CONFIG["classes"],
        test_manifest=test_manifest,
    )
    evaluation = evaluate_model(
        final_checkpoint,
        evaluation_dataset_yaml,
        split="test",
        imgsz=selected_imgsz,
        device=TRAINING_CONFIG["device"],
    )
    final_overall_df = pd.DataFrame([evaluation["overall"]])
    final_class_metrics_df = evaluation["class_metrics"]
    final_overall_df.to_csv(RUN.csv / "final_test_overall_metrics.csv", index=False)
    if not final_class_metrics_df.empty:
        final_class_metrics_df.to_csv(
            RUN.csv / "final_test_class_metrics.csv",
            index=False,
        )
    if evaluation["confusion_matrix"] is not None:
        confusion_figure_path = save_confusion_matrix(
            evaluation["confusion_matrix"],
            CONFIG["classes"].values(),
            RUN.figures / "final_test_confusion_matrix.png",
        )
    prediction_figure_path = save_prediction_grid(
        final_checkpoint,
        test_manifest,
        RUN.figures / "final_test_prediction_samples.png",
        imgsz=selected_imgsz,
        device=TRAINING_CONFIG["device"],
        conf=float(TRAINING_CONFIG["final_prediction_conf"]),
        max_images=6,
    )
    display(final_overall_df)
    display(final_class_metrics_df)


#### Observation

TODO after execution:

- Report untouched-test overall and per-class weaknesses.
- Describe visible failure modes from the bounded prediction grid.
- Never rerun epoch calibration or resolution selection from these test results.


## Export and apply retention policy

Export only configured deployment formats. Keep the descriptive refit
checkpoint, then remove this run's calibration/refit native work directories
and pretrained cache when intermediate retention is disabled.


In [ ]:
# Purpose: export the retained refit model and remove redundant current-run internals.
# Inputs: retained checkpoint, configured export formats, and keep_intermediates policy.
# Guard: export failures are reported without deleting the retained PyTorch checkpoint.
# Output: optional deployment model(s), one compact export CSV, and a clean models folder.
export_report_df = pd.DataFrame()
export_summary_path = None
if not DRY_RUN and final_checkpoint is not None:
    export_report_df = export_final_model(
        final_checkpoint,
        TRAINING_CONFIG["final_export_formats"],
        RUN.models,
    )
    export_summary_path = RUN.csv / "final_export_summary.csv"
    export_report_df.to_csv(export_summary_path, index=False)
    if not bool(TRAINING_CONFIG["keep_intermediates"]):
        clean_training_intermediates(RUN.models)
    display(export_report_df)


#### Observation

TODO after execution: record retained checkpoint size, deployment format
availability, and any machine-dependent export failure.


## Finalize the immutable calibrated-refit report

Persist the validation-derived epoch, Notebook 06 decision chain, both
training contracts, test metrics, and retained paths in this run's `run.yaml`.


In [ ]:
# Purpose: close the final run with auditable calibration, refit, test, and artifact evidence.
# Inputs: frozen manifests, selected epoch, summaries, test results, figures, exports, and upstream metadata.
# Guard: update only this newly created run and never reinterpret or overwrite a previous test run.
# Output: final run.yaml plus concise run ID, locked epoch, resolution, and completion status.
update_run_metadata(
    RUN,
    {
        "status": "dry_run" if DRY_RUN else "completed",
        "selected_model": str(selected_model),
        "selected_experiment": selected_experiment,
        "selected_resolution_profile": str(selected_resolution_profile),
        "selected_imgsz": selected_imgsz,
        "selected_batch": selected_batch,
        "epoch_policy": "calibrate_on_train_val_then_refit_train_plus_val",
        "epoch_selection_source": epoch_selection_source,
        "locked_refit_epochs": locked_refit_epochs,
        "epoch_selection_validation_metrics": {
            key: value
            for key, value in epoch_selection.items()
            if key not in {"best_epoch", "selection_metric"}
        },
        "final_experiment_override": FINAL_EXPERIMENT_OVERRIDE,
        "experiment_selection_source": experiment_selection_source,
        "source_resolution_run": RESOLUTION_RUN.run_id,
        "source_ablation_run": resolution_metadata.get("source_ablation_run"),
        "calibration_dataset_yaml": str(calibration_dataset_yaml),
        "refit_dataset_yaml": str(refit_dataset_yaml),
        "evaluation_dataset_yaml": str(evaluation_dataset_yaml)
        if evaluation_dataset_yaml
        else None,
        "calibration_train_manifest_sha256": manifest_sha256(
            calibration_train_manifest
        ),
        "calibration_val_manifest_sha256": manifest_sha256(
            calibration_val_manifest
        ),
        "refit_train_plus_val_manifest_sha256": manifest_sha256(
            refit_training_manifest
        ),
        "test_manifest_sha256": manifest_sha256(test_manifest),
        "test_pair_content_sha256": test_content_sha256,
        "calibration_summary_csv": str(calibration_summary_path),
        "refit_summary_csv": str(refit_summary_path),
        "final_checkpoint": str(final_checkpoint) if final_checkpoint else None,
        "test_overall_metrics": final_overall_df.to_dict(orient="records"),
        "test_class_metrics_csv": str(RUN.csv / "final_test_class_metrics.csv")
        if not final_class_metrics_df.empty
        else None,
        "confusion_matrix_figure": str(confusion_figure_path)
        if confusion_figure_path
        else None,
        "prediction_samples_figure": str(prediction_figure_path)
        if prediction_figure_path
        else None,
        "export_summary_csv": str(export_summary_path)
        if export_summary_path
        else None,
    },
)
print(f"Final calibrated-refit run: {RUN.run_id}")
print(f"Locked refit epochs: {locked_refit_epochs}")
print(
    f"Resolution: {selected_resolution_profile} "
    f"({selected_imgsz}px, batch={selected_batch})"
)
print(f"Status: {'dry_run' if DRY_RUN else 'completed'}")


#### Observation

TODO after execution: write the exact immutable run ID, validation-selected
epoch, deployment recommendation, known limitations, and whether this locked
configuration should be promoted.
